## Process GloFAS ensemble reforecast data to identify extreme rainfall events in Dublin 

In [1]:
import glob
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging
from pathlib import Path
import time

### START FROM HERE

- Read in nc file 
- Rank the run/lead time combinations to find the largest spread 
- Cross-check with GloFAS reanalysis to see whether these were real events 
- Return the top 5 dates 
- Address the missing data (i.e. how to process and calculate spread)

In [2]:
file_path = '/mnt/metdata/W25-2348/glofas/processed_spread'
files = sorted(glob.glob(f"{file_path}/glofas_spread_*.nc"))
print(files)
records = []
for f in files:
    ds = xr.open_dataset(f)
    da = ds["spread_maxmin"]   # dims time, step
    # convert to dataframe for this file
    df = da.to_dataframe(name="spread").reset_index()   # time, step, spread
    df = df.dropna(subset=["spread"])
    df["year_file"] = f
    records.append(df)

df_all = pd.concat(records, ignore_index=True)

# convert step to hours
if pd.api.types.is_timedelta64_dtype(df_all["step"].dtype):
    df_all["lead_hours"] = df_all["step"].dt.total_seconds() / 3600.0

# sort and save
df_sorted = df_all.sort_values("spread", ascending=False).reset_index(drop=True)
df_sorted.head(50).to_csv(f"{file_path}/glofas_ensemble_reforecast_spread_top50.csv", index=False)

['/mnt/metdata/W25-2348/glofas/processed_spread/glofas_spread_2003.nc', '/mnt/metdata/W25-2348/glofas/processed_spread/glofas_spread_2005.nc', '/mnt/metdata/W25-2348/glofas/processed_spread/glofas_spread_2006.nc', '/mnt/metdata/W25-2348/glofas/processed_spread/glofas_spread_2007.nc', '/mnt/metdata/W25-2348/glofas/processed_spread/glofas_spread_2008.nc', '/mnt/metdata/W25-2348/glofas/processed_spread/glofas_spread_2009.nc', '/mnt/metdata/W25-2348/glofas/processed_spread/glofas_spread_2010.nc', '/mnt/metdata/W25-2348/glofas/processed_spread/glofas_spread_2011.nc', '/mnt/metdata/W25-2348/glofas/processed_spread/glofas_spread_2012.nc', '/mnt/metdata/W25-2348/glofas/processed_spread/glofas_spread_2014.nc', '/mnt/metdata/W25-2348/glofas/processed_spread/glofas_spread_2015.nc', '/mnt/metdata/W25-2348/glofas/processed_spread/glofas_spread_2016.nc', '/mnt/metdata/W25-2348/glofas/processed_spread/glofas_spread_2017.nc', '/mnt/metdata/W25-2348/glofas/processed_spread/glofas_spread_2018.nc', '/mnt

### Calculate largest ensemble spread per lead time

In [3]:
largest_spread_index_per_lead_time = df_all.groupby("step")["spread"].idxmax()
largest_spread_per_lead_time = df_all.loc[largest_spread_index_per_lead_time].sort_values("step")
largest_spread_per_lead_time.to_csv(f'{file_path}/largest_ensemble_spread_per_lead_time.csv', index=False)

### Top 5 dates per lead time

In [4]:
df_all['lead_days'] = df_all['step'].dt.days
df_all['lead_str'] = df_all['lead_days'].astype(str) + 'd'
for lead_time in df_all['lead_str'].unique():
    df_all[df_all['lead_str'] == lead_time].sort_values("spread", ascending=False).head(5).reset_index(drop=True).to_csv(f'{file_path}/top5_per_lead_time_{lead_time}.csv')